In [ ]:
# model2_survey_full.py
# Multiclass (0/1/2) per product + per-label weights + calibration + precision-tuned thresholds + gated Top-k

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    hamming_loss, accuracy_score, jaccard_score, f1_score,
    precision_score, recall_score, coverage_error,
    label_ranking_average_precision_score, precision_recall_curve
)
from sklearn.calibration import CalibratedClassifierCV
import lightgbm as lgb

# -------------------------
# Config (tweak as desired)
# -------------------------
RANDOM_STATE = 42

# Utility score = w2*p2 + w1*p1
UTILITY_W2 = 2.5   # weight for class-2 (high interest)
UTILITY_W1 = 0.5   # weight for class-1 (low interest)

# Global gating criteria (reduce rating-0 leakage)
SCORE_FLOOR = 0.50   # global minimum utility score
P2_MIN      = 0.35   # minimum p2 (confidence of rating=2)

# Per-label precision target for threshold tuning on utility score
MIN_POS_SUPPORT = 25         # only tune label if it has sufficient positives
DEFAULT_THR     = 0.80       # default label threshold on utility score

# LightGBM (same across labels; per-label class_weight handled below)
LGBM_PARAMS = dict(
    objective='multiclass',
    num_class=3,
    boosting_type='gbdt',
    n_estimators=147,
    max_depth=14,
    num_leaves=85,
    learning_rate=0.1985,
    min_child_samples=32,
    subsample=0.6620,
    colsample_bytree=0.9193,
    reg_alpha=0.4385,
    reg_lambda=0.6720,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

# -------------------------
# Helper functions
# -------------------------
def compute_scores(p1, p2, w1=UTILITY_W1, w2=UTILITY_W2):
    """Utility score used for ranking and thresholding."""
    return w2 * p2 + w1 * p1

def compute_class_weights_for_label(y):
    """
    Build per-class weights for a single label (0/1/2).
    We favor rating=2 slightly more than 1.
    """
    total = len(y)
    c0 = int(np.sum(y == 0))
    c1 = int(np.sum(y == 1))
    c2 = int(np.sum(y == 2))

    w0 = 1.0
    w1 = (total / (3 * c1)) if c1 > 0 else 1.0
    w2 = (total / (2 * c2)) if c2 > 0 else 1.0

    # mild cap to avoid extreme weights
    w1 = float(np.clip(w1, 0.5, 5.0))
    w2 = float(np.clip(w2, 0.5, 6.0))
    return {0: w0, 1: w1, 2: w2}

def train_per_label_models_with_calibration(X_tr, X_val, Y_tr_mc, Y_val_mc, feature_names):
    """
    Train one 3-class LGBM per product with per-label class_weight,
    then calibrate (Platt) each model on the validation split.
    Returns a list of calibrated estimators in the order of columns in Y_tr_mc.
    """
    models = []
    for j, col in enumerate(Y_tr_mc.columns):
        y_tr  = Y_tr_mc.iloc[:, j].astype(int).values
        y_val = Y_val_mc.iloc[:, j].astype(int).values

        class_weight = compute_class_weights_for_label(y_tr)

        base = lgb.LGBMClassifier(**LGBM_PARAMS, class_weight=class_weight)
        base.fit(X_tr, y_tr, feature_name=feature_names)

        # choose calibration: isotonic if enough positives (>=1000), else sigmoid
        pos_tr = np.sum(y_tr >= 1)
        cal_method = 'isotonic' if pos_tr >= 1000 else 'sigmoid'

        calib = CalibratedClassifierCV(base, method=cal_method, cv='prefit')
        calib.fit(X_val, y_val)
        models.append(calib)
    return models


def predict_p1_p2(models, X):
    """
    Get p1 and p2 matrices from a list of (calibrated) per-label estimators.
    Returns:
      p1: (n_samples, n_labels) probabilities for class 1
      p2: (n_samples, n_labels) probabilities for class 2
    """
    p1_cols, p2_cols = [], []
    for est in models:
        proba = est.predict_proba(X)  # (n, 3)
        p1_cols.append(proba[:, 1])
        p2_cols.append(proba[:, 2])
    p1 = np.column_stack(p1_cols)
    p2 = np.column_stack(p2_cols)
    return p1, p2

def tune_thresholds_precision(scores_val, y_val_pos, target_precision=TARGET_PRECISION,
                              min_pos=MIN_POS_SUPPORT, default_thr=DEFAULT_THR):
    """
    Per-label threshold tuning on utility scores to meet a precision target on (rating>=1).
    """
    L = scores_val.shape[1]
    thrs = np.full(L, default_thr, dtype=float)

    for j in range(L):
        y_true = y_val_pos[:, j]
        if y_true.sum() < min_pos:
            continue

        precision, recall, thresholds = precision_recall_curve(y_true, scores_val[:, j])
        precision = precision[1:]; recall = recall[1:]  # align to thresholds

        ok = np.where(precision >= target_precision)[0]
        if ok.size > 0:
            thrs[j] = thresholds[ok[0]]
        else:
            if thresholds.size > 0:
                f1 = (2 * precision * recall) / np.clip(precision + recall, 1e-9, None)
                thrs[j] = thresholds[np.argmax(f1)]
    return thrs

def gated_topk(scores, p2, label_thrs, k=3, floor=SCORE_FLOOR, p2_min=P2_MIN, enforce_exact_k=True):
    """
    Per-row selection:
      keep labels with score >= max(label_thr, floor) AND (p2>=p2_min OR score>=label_thr),
      sort by score desc, take up to k, then backfill if needed.
    """
    n, L = scores.shape
    out = np.zeros((n, k), dtype=int)

    # precompute adaptive floor per label
    adaptive_floor = np.maximum(floor, 0.7 * label_thrs)

    for i in range(n):
        s = scores[i]
        p2row = p2[i]
        # per-label gate
        keep = (s >= np.maximum(label_thrs, adaptive_floor)) & ((p2row >= p2_min) | (s >= label_thrs))
        cand = np.where(keep)[0]
        cand_sorted = cand[np.argsort(s[cand])[::-1]]

        if cand_sorted.size >= k:
            top = cand_sorted[:k]
        else:
            # Optional strict mode: comment OUT the next two lines to allow <k results
            rest = np.setdiff1d(np.argsort(s)[::-1], cand_sorted, assume_unique=False)
            rest = rest[s[rest] >= floor]

            needed = k - cand_sorted.size
            top = np.concatenate([cand_sorted, rest[:needed]])
            if enforce_exact_k and top.size < k:
                rest2 = np.setdiff1d(np.argsort(s)[::-1], top, assume_unique=False)
                top = np.concatenate([top, rest2[:(k - top.size)]])
        out[i] = top[:k]
    return out


def topk_hit_rate(Y_true_bin, idx):
    """
    Hit if any of the true positives is in the chosen top-k.
    Only count rows that have at least one true positive.
    """
    hits = 0; total = 0
    for i in range(Y_true_bin.shape[0]):
        true_pos = np.where(Y_true_bin[i] == 1)[0]
        if true_pos.size == 0:
            continue
        total += 1
        if np.intersect1d(true_pos, idx[i]).size > 0:
            hits += 1
    return hits / max(total, 1)

# -------------------------
# Load & preprocess data
# -------------------------
print("Loading data and building Model 2 (Survey):")
df = pd.read_csv("adviser_survey.csv")

# If present, drop YearsExperience (you can re-add later if you like)
if 'YearsExperience' in df.columns:
    df = df.drop(columns=['YearsExperience'])

# Identify demographics vs product rating columns
demographic_cols = ['AgeGroup', 'Gender', 'MaritalStatus', 'IncomeBracket']
demographic_cols = [c for c in demographic_cols if c in df.columns]  # keep only existing ones
product_cols = [c for c in df.columns if c not in demographic_cols]

# Convert product ratings to numeric 0/1/2
for col in product_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
    df[col] = df[col].clip(lower=0, upper=2)

# One-hot encode demographics
df_encoded = pd.get_dummies(df, columns=demographic_cols, drop_first=False)

# Features and labels
feature_cols = [c for c in df_encoded.columns if c not in product_cols]
X_all = df_encoded[feature_cols].astype(float)
Y_mc_all = df_encoded[product_cols].astype(int)    # multiclass labels (0/1/2)
Y_bin_all = (Y_mc_all.values >= 1).astype(int)     # binary relevance for some metrics

feature_names = list(X_all.columns)
label_names = list(Y_mc_all.columns)

# -------------------------
# Split: Train / Val / Test
# -------------------------
X_train, X_test, Y_train_mc, Y_test_mc = train_test_split(
    X_all, Y_mc_all, test_size=0.20, random_state=RANDOM_STATE
)
X_tr, X_val, Y_tr_mc, Y_val_mc = train_test_split(
    X_train, Y_train_mc, test_size=0.20, random_state=RANDOM_STATE
)

Y_test_bin = (Y_test_mc.values >= 1).astype(int)

# -------------------------
# Train per-label models + Calibrate on Val
# -------------------------
models = train_per_label_models_with_calibration(
    X_tr, X_val, Y_tr_mc, Y_val_mc, feature_names
)

# -------------------------
# Tune per-label thresholds on Val (precision targeting on utility score)
# -------------------------
p1_val, p2_val = predict_p1_p2(models, X_val)
scores_val = compute_scores(p1_val, p2_val, UTILITY_W1, UTILITY_W2)
Y_val_pos = (Y_val_mc.values >= 1).astype(int)

label_thresholds = tune_thresholds_precision(
    scores_val, Y_val_pos,
    target_precision=TARGET_PRECISION,
    min_pos=MIN_POS_SUPPORT,
    default_thr=DEFAULT_THR
)

# -------------------------
# Evaluate on Test
# -------------------------
p1_test, p2_test = predict_p1_p2(models, X_test)
scores_test = compute_scores(p1_test, p2_test, UTILITY_W1, UTILITY_W2)

# Thresholded multilabel predictions (for classification metrics)
Y_pred_bin = (
    (scores_test >= np.maximum(label_thresholds, SCORE_FLOOR)) &
    ((p2_test >= P2_MIN) | (scores_test >= label_thresholds))
).astype(int)

# Gated Top-k indices (k = 1/3/5)
topk1_idx = gated_topk(scores_test, p2_test, label_thresholds, k=1,
                       floor=SCORE_FLOOR, p2_min=P2_MIN, enforce_exact_k=True)
topk3_idx = gated_topk(scores_test, p2_test, label_thresholds, k=3,
                       floor=SCORE_FLOOR, p2_min=P2_MIN, enforce_exact_k=True)
topk5_idx = gated_topk(scores_test, p2_test, label_thresholds, k=5,
                       floor=SCORE_FLOOR, p2_min=P2_MIN, enforce_exact_k=True)

# -------------------------
# Metrics
# -------------------------
print("\n=== MODEL 2 (Survey) – Multiclass + Per-Label Weights + Calibration + Gated Top-k ===")

print("\nTop-k Accuracy (binary relevance ≥1):")
print(f"  Top-1: {topk_hit_rate(Y_test_bin, topk1_idx):.4f} ({topk_hit_rate(Y_test_bin, topk1_idx)*100:.2f}%)")
print(f"  Top-3: {topk_hit_rate(Y_test_bin, topk3_idx):.4f} ({topk_hit_rate(Y_test_bin, topk3_idx)*100:.2f}%)")
print(f"  Top-5: {topk_hit_rate(Y_test_bin, topk5_idx):.4f} ({topk_hit_rate(Y_test_bin, topk5_idx)*100:.2f}%)")

print("\nClassification metrics (thresholded by per-label tuned thresholds):")
print(f"  Hamming Loss:    {hamming_loss(Y_test_bin, Y_pred_bin):.4f}")
print(f"  Subset Accuracy: {accuracy_score(Y_test_bin, Y_pred_bin):.4f}")
print(f"  F1 (micro):      {f1_score(Y_test_bin, Y_pred_bin, average='micro'):.4f}")
print(f"  F1 (macro):      {f1_score(Y_test_bin, Y_pred_bin, average='macro'):.4f}")
print(f"  F1 (weighted):   {f1_score(Y_test_bin, Y_pred_bin, average='weighted'):.4f}")
print(f"  Precision (w):   {precision_score(Y_test_bin, Y_pred_bin, average='weighted', zero_division=0):.4f}")
print(f"  Recall (w):      {recall_score(Y_test_bin, Y_pred_bin, average='weighted'):.4f}")
print(f"  Jaccard (w):     {jaccard_score(Y_test_bin, Y_pred_bin, average='weighted'):.4f}")
print(f"  Coverage Error:  {coverage_error(Y_test_bin, scores_test):.4f}")
print(f"  LRAP:            {label_ranking_average_precision_score(Y_test_bin, scores_test):.4f}")

# Prioritization (how well we score r=2 > r=1 > r=0)
r2 = scores_test[Y_test_mc.values == 2].mean() if (Y_test_mc.values == 2).any() else 0.0
r1 = scores_test[Y_test_mc.values == 1].mean() if (Y_test_mc.values == 1).any() else 0.0
r0 = scores_test[Y_test_mc.values == 0].mean() if (Y_test_mc.values == 0).any() else 0.0

print(f"\nPrioritization (scores are {UTILITY_W2}*p2 + {UTILITY_W1}*p1):")
print(f"  Avg score r=2:   {r2:.4f}")
print(f"  Avg score r=1:   {r1:.4f}")
print(f"  Avg score r=0:   {r0:.4f}")
print(f"  Ratio 2/1:       {(r2 / max(r1, 1e-9)):.3f}x")
print(f"  Ratio 1/0:       {(r1 / max(r0, 1e-9)):.3f}x")

# Top-3 composition by true rating
rating_in_top3 = {0: 0, 1: 0, 2: 0}
total_top3 = topk3_idx.shape[0] * 3
for i in range(topk3_idx.shape[0]):
    for idx in topk3_idx[i]:
        true_rating = Y_test_mc.iloc[i, idx]
        rating_in_top3[int(true_rating)] += 1

print("\nTop-3 composition by true rating:")
print(f"  r=2: {rating_in_top3[2]:4d} ({rating_in_top3[2]/total_top3*100:.1f}%)")
print(f"  r=1: {rating_in_top3[1]:4d} ({rating_in_top3[1]/total_top3*100:.1f}%)")
print(f"  r=0: {rating_in_top3[0]:4d} ({rating_in_top3[0]/total_top3*100:.1f}%)")

# Example client (first in test set)
i = 0
scores_i = scores_test[i]
p2_i = p2_test[i]
thr_i = label_thresholds
order = topk3_idx[i]

print("\nExample client (first in test set) – Top-3 with gating:")
for rank, lbl_idx in enumerate(order, 1):
    print(f"  {rank}. {label_names[lbl_idx]}  (score={scores_i[lbl_idx]:.3f}, thr={thr_i[lbl_idx]:.2f})")

# Optional: quick summary of tuned thresholds
thr_summary = pd.Series(label_thresholds, index=label_names)
q = thr_summary.quantile([0.1,0.25,0.5,0.75,0.9]).round(3)
print("\nPer-label threshold summary (utility score):")
print(q.to_string())
print(f"Min={thr_summary.min():.3f}  Mean={thr_summary.mean():.3f}  Max={thr_summary.max():.3f}")


Loading data and building Model 2 (Survey):

=== MODEL 2 (Survey) – Multiclass + Per-Label Weights + Calibration + Gated Top-k ===

Top-k Accuracy (binary relevance ≥1):
  Top-1: 0.8748 (87.48%)
  Top-3: 0.9569 (95.69%)
  Top-5: 0.9717 (97.17%)

Classification metrics (thresholded by per-label tuned thresholds):
  Hamming Loss:    0.2046
  Subset Accuracy: 0.0000
  F1 (micro):      0.4228
  F1 (macro):      0.1296
  F1 (weighted):   0.2857
  Precision (w):   0.3699
  Recall (w):      0.3119
  Jaccard (w):     0.2277
  Coverage Error:  28.4573
  LRAP:            0.6888

Prioritization (scores are 2.5*p2 + 0.5*p1):
  Avg score r=2:   0.7788
  Avg score r=1:   0.4598
  Avg score r=0:   0.2339
  Ratio 2/1:       1.694x
  Ratio 1/0:       1.966x

Top-3 composition by true rating:
  r=2: 1177 (52.3%)
  r=1:  492 (21.9%)
  r=0:  581 (25.8%)

Example client (first in test set) – Top-3 with gating:
  1. Integrated_Shield  (score=1.865, thr=1.67)
  2. RP_WL_Protection_with_multiplier  (score=1.2

In [5]:
# === model2_ops_bundle.py ===
import numpy as np
import pandas as pd
import joblib
from dataclasses import dataclass, asdict

@dataclass
class Model2Bundle:
    models: list                 # list of CalibratedClassifierCV per label
    feature_names: list          # one-hot feature columns used in training
    label_names: list            # product label order
    label_thresholds: np.ndarray # per-label utility thresholds
    utility_w2: float            # weight for class-2 prob
    utility_w1: float            # weight for class-1 prob
    score_floor: float           # global min utility score
    p2_min: float                # min prob(class=2)
    demographic_cols: list       # raw demographic columns (for one-hot at inference)

    # ---------- internal helpers ----------
    def _ensure_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        # If raw demographic columns are present, one-hot them to match training
        if any(c in df.columns for c in self.demographic_cols):
            df = pd.get_dummies(df, columns=self.demographic_cols, drop_first=False)
        # Add missing training columns with 0, then order
        for col in self.feature_names:
            if col not in df.columns:
                df[col] = 0.0
        X = df[self.feature_names].astype(float)
        return X

    def _predict_p1_p2(self, X: pd.DataFrame):
        p1_list, p2_list = [], []
        for est in self.models:
            proba = est.predict_proba(X)  # shape (n, 3)
            p1_list.append(proba[:, 1])
            p2_list.append(proba[:, 2])
        p1 = np.column_stack(p1_list)
        p2 = np.column_stack(p2_list)
        return p1, p2

    def _compute_scores(self, p1, p2):
        return self.utility_w2 * p2 + self.utility_w1 * p1

    def _gated_topk(self, scores, p2, k=3, enforce_exact_k=True):
        n, L = scores.shape
        out = np.zeros((n, k), dtype=int)
        thr = np.asarray(self.label_thresholds)
        adaptive_floor = np.maximum(self.score_floor, 0.7 * thr)

        for i in range(n):
            s = scores[i]; p2i = p2[i]
            keep = (s >= np.maximum(thr, adaptive_floor)) & ((p2i >= self.p2_min) | (s >= thr))
            cand = np.where(keep)[0]
            cand_sorted = cand[np.argsort(s[cand])[::-1]]

            if cand_sorted.size >= k:
                top = cand_sorted[:k]
            else:
                # backfill by score >= global floor, then anything if still short
                rest = np.setdiff1d(np.argsort(s)[::-1], cand_sorted, assume_unique=False)
                rest = rest[s[rest] >= self.score_floor]
                top = np.concatenate([cand_sorted, rest[:max(0, k-cand_sorted.size)]])
                if enforce_exact_k and top.size < k:
                    rest2 = np.setdiff1d(np.argsort(s)[::-1], top, assume_unique=False)
                    top = np.concatenate([top, rest2[:(k - top.size)]])
            out[i] = top[:k]
        return out

    # ---------- public API ----------
    def predict_topk(self, df_or_X, k=3):
        X = self._ensure_features(df_or_X) if isinstance(df_or_X, pd.DataFrame) \
            else pd.DataFrame(df_or_X, columns=self.feature_names)
        p1, p2 = self._predict_p1_p2(X)
        scores = self._compute_scores(p1, p2)
        idx = self._gated_topk(scores, p2, k=k, enforce_exact_k=True)

        rows = []
        thr = np.asarray(self.label_thresholds)
        for i in range(idx.shape[0]):
            for rank, j in enumerate(idx[i], 1):
                rows.append({
                    "row_id": i,
                    "rank": rank,
                    "label": self.label_names[j],
                    "score": float(scores[i, j]),
                    "p2": float(p2[i, j]),
                    "threshold": float(thr[j]),
                    "gated": bool(scores[i, j] >= max(thr[j], self.score_floor) and
                                  ((p2[i, j] >= self.p2_min) or (scores[i, j] >= thr[j])))
                })
        return pd.DataFrame(rows)

    def predict_binary(self, df_or_X):
        X = self._ensure_features(df_or_X) if isinstance(df_or_X, pd.DataFrame) \
            else pd.DataFrame(df_or_X, columns=self.feature_names)
        p1, p2 = self._predict_p1_p2(X)
        scores = self._compute_scores(p1, p2)
        thr = np.asarray(self.label_thresholds)
        Y = ((scores >= np.maximum(thr, self.score_floor)) &
             ((p2 >= self.p2_min) | (scores >= thr))).astype(int)
        return pd.DataFrame(Y, columns=self.label_names)

    def save(self, path: str):
        joblib.dump({
            "models": self.models,
            "feature_names": self.feature_names,
            "label_names": self.label_names,
            "label_thresholds": np.asarray(self.label_thresholds),
            "utility_w2": self.utility_w2,
            "utility_w1": self.utility_w1,
            "score_floor": self.score_floor,
            "p2_min": self.p2_min,
            "demographic_cols": self.demographic_cols,
        }, path)

    @staticmethod
    def load(path: str) -> "Model2Bundle":
        obj = joblib.load(path)
        return Model2Bundle(**obj)


In [6]:
bundle = Model2Bundle(
    models=models,
    feature_names=feature_cols,
    label_names=label_names,
    label_thresholds=label_thresholds,
    utility_w2=UTILITY_W2,
    utility_w1=UTILITY_W1,
    score_floor=SCORE_FLOOR,
    p2_min=P2_MIN,
    demographic_cols=demographic_cols  # from your script
)

# save bundle for deployment
bundle.save("model2_bundle.joblib")

# quick sanity check on your held-out test set (already encoded)
print("\nSample Top-3 predictions (first 5 rows):")
print(bundle.predict_topk(X_test.iloc[:5], k=3))

# later, in any service/batch job:
# loaded = Model2Bundle.load("model2_bundle.joblib")
# preds = loaded.predict_topk(new_clients_df, k=3)  # raw demo cols ok; it will one-hot + align



Sample Top-3 predictions (first 5 rows):
    row_id  rank                             label     score        p2  \
0        0     1                 Integrated_Shield  1.865171  0.720495   
1        0     2  RP_WL_Protection_with_multiplier  1.286606  0.455971   
2        0     3      Early_Stage_Critical_Illness  1.053837  0.360936   
3        1     1                 Integrated_Shield  1.865171  0.720495   
4        1     2  RP_WL_Protection_with_multiplier  1.852365  0.676182   
5        1     3      Early_Stage_Critical_Illness  1.077295  0.376223   
6        2     1                 Integrated_Shield  1.865171  0.720495   
7        2     2  RP_WL_Protection_with_multiplier  1.228827  0.436267   
8        2     3      Early_Stage_Critical_Illness  1.097979  0.393346   
9        3     1                 Integrated_Shield  1.786374  0.688554   
10       3     2           Education_Funding_Plans  1.230786  0.477433   
11       3     3  RP_WL_Protection_with_multiplier  1.225662  0.434372